## base_model--> non_instruction_model--> instruction_model--> preference_model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [ ]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
import zipfile
import os
# Path to your zip file
zip_path = "/content/tinyllama-instruction.zip"

# Extract all files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
model_path = "/content/checkpoint-3"

In [ ]:
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

In [ ]:
prompt = "Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry."

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Now lets start with prefrence base tuning or preference based alignment

In [ ]:
!pip install -U trl

In [ ]:
!pip install -U bitsandbytes

In [ ]:
from trl import DPOTrainer
from transformers import AutoTokenizer,  AutoModelForCausalLM, TrainingArguments
from peft import PeftModel
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch

In [ ]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
instruction_checkpoint = "/content/checkpoint-3"

In [ ]:
# Load dataset
dataset = load_dataset("csv", data_files="/content/pharma_preference_data.csv")["train"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

get_peft_model() → Create a new LoRA during training

PeftModel.from_pretrained() → Load an already-trained LoRA for inference or further training

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [ ]:
# pref_model_lora = get_peft_model(instruction_model, lora_config)

In [ ]:
base_model

In [ ]:
#STEP A: Load base
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    load_in_8bit=True,
    device_map="auto"
)

In [ ]:
#STEP B: Load Instruction LoRA + merge
model = PeftModel.from_pretrained(model, instruction_checkpoint)

In [ ]:
model = model.merge_and_unload()

In [ ]:
#STEP C: Attach NEW LoRA for preference
pref_model_lora = get_peft_model(model, lora_config)

| Stage           | What You Should Do                       | Wrong Way (you did)      |
| --------------- | ---------------------------------------- | ------------------------ |
| Non-Instruction | Base + LoRA                              | ✔ correct                |
| Instruction     | Base + **merge(stage1 LoRA)** + NEW LoRA | ❌ “LoRA on LoRA”         |
| Preference      | Base + **merge(stage2 LoRA)** + NEW LoRA | ❌ “LoRA on LoRA on LoRA” |


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
from trl import DPOTrainer, DPOConfig

In [ ]:
dpo_args = DPOConfig(
    output_dir="./tinyllama-preference-alignment",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    beta=0.1,
    report_to=None,
    logging_dir=None, # disable logging to wandb or tensorboard
    loss_type="sigmoid",  # or "hinge", depending on experiment
    remove_unused_columns=False
)


In [ ]:
trainer = DPOTrainer(
    model=pref_model_lora,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset,
    processing_class=tokenizer,   # instead of tokenizer argument
    # you can pass data_collator if needed,
    # optionally eval_dataset etc.
)

In [ ]:
trainer.train()

### Testing with Non-Instruction Model

In [ ]:
question = "Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment."

In [ ]:
import zipfile
import os
# Path to your zip file
zip_path = "/content/tinyllama-non-instruction.zip"

# Extract all files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
model_path = "/content/checkpoint-5"
non_instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

In [ ]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [ ]:
outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Testing with Instruction-Fine-Tuned Model

In [ ]:
model_path = "/content/checkpoint-3"
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

In [ ]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [ ]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)


In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Testing with DPO (Preference-Aligned) Model

In [ ]:
model_path = "/content/tinyllama-preference-alignment/checkpoint-1"

In [ ]:
preference_aligned_model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float16)

In [ ]:
preference_aligned_model.to("cuda")

In [ ]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [ ]:
outputs = preference_aligned_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))